# Stage 6 — A/B/C Variant Comparison (Overqualified Senior Candidate Resume)

**Represents 5 of the 20 required test cases** (this resume × 5 jobs). Fourth and final resume-based batch (Trixie Mok, Alex Chen, John Doe, Alex Mercer), 5 test cases each, totalling 20.

Resume tested: `overqualified_senior_candidate_resume_example.pdf` — Alex Mercer, a Senior Lead Engineer with over a decade of real, deep front-end expertise (React, TypeScript, Next.js, CI/CD, testing), explicitly stepping back down into a mid-level Individual Contributor role. A different candidate archetype from the other three: not a weak or missing background, but a strong, *real, narrowly-scoped* one (front-end specifically) tested against jobs mostly outside that specialty.

- **A — Minimal LLM**: bare prompt, "Rewrite this resume for this job."
- **B — Simplified system**: "Extract relevant skills and tailor this resume to the job" — no schema, no anti-fabrication instructions.
- **C — Full system**: the actual deployed app logic (`JobPortalService._generate_tailored_resume`), structured JSON schema, evidence required per claim, explicit instruction never to invent credentials/employers/dates/metrics/skills, plus the deterministic `_enforce_fidelity` post-check.

Same 5 jobs and 4 criteria as the other three resume tests, for direct comparability. This resume was tested directly against the already-fixed code — there is no pre-fix baseline for it, unlike the other three resumes.

In [1]:
import json
from pathlib import Path

HERE = Path(".")
naive = json.loads((HERE / "variant_comparison_results.json").read_text(encoding="utf-8"))
grounded = json.loads((HERE / "variant_comparison_results_grounded_judge.json").read_text(encoding="utf-8"))

CRITERIA = ["grounding", "personalization", "correctness", "clarity"]
print(f"Loaded {len(naive)} test cases (naive judge) and {len(grounded)} (grounding-gated judge)")

Loaded 5 test cases (naive judge) and 5 (grounding-gated judge)


## Round 1: Naive LLM-judge scoring

The judge was told to score 1-5 on each criterion, with no explicit instruction about how to weigh fabrication vs. fluency.

In [2]:
def print_table(results, score_key):
    header = f"{'Job':<22}{'Variant':<8}" + "".join(f"{c[:10]:<12}" for c in CRITERIA) + "avg"
    print(header)
    print("-" * len(header))
    totals = {v: {c: [] for c in CRITERIA} for v in "ABC"}
    for entry in results:
        scores_block = entry[score_key]
        for v in "ABC":
            row = scores_block[v]
            vals = [row[c]["score"] for c in CRITERIA]
            avg = sum(vals) / len(vals)
            for c, val in zip(CRITERIA, vals):
                totals[v][c].append(val)
            print(f"{entry['job_title'][:21]:<22}{v:<8}" + "".join(f"{val:<12}" for val in vals) + f"{avg:.2f}")
    print()
    print("OVERALL AVERAGES")
    for v in "ABC":
        per_c = {c: sum(totals[v][c]) / len(totals[v][c]) for c in CRITERIA}
        overall = sum(per_c.values()) / len(per_c)
        print(f"  {v}: " + ", ".join(f"{c}={val:.2f}" for c, val in per_c.items()) + f"  -> overall={overall:.2f}")

print_table(naive, "scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       2           4           3           4           3.25
Software Engineer     B       4           4           4           4           4.00
Software Engineer     C       5           3           5           3           4.00
Machine Learning      A       1           2           2           3           2.00
Machine Learning      B       4           3           4           4           3.75
Machine Learning      C       5           1           5           5           4.00
DevOps Engineer       A       1           4           2           4           2.75
DevOps Engineer       B       2           4           3           4           3.25
DevOps Engineer       C       5           2           5           3           3.75
Database Administrato A       2           4           3           4           3.25
Databa

Even the naive judge ranks C highest here, largely on the strength of its perfect grounding and correctness — but C's personalization score is the lowest of any variant across all four resume tests so far. That's consistent with the resume itself: a genuinely senior, narrowly-specialized front-end engineer has real, deep experience the job requirements for DevOps/ML/DB roles simply don't call for, so an honest tailoring pass reads as poorly matched rather than as compensating with invented breadth.

## Round 2: Grounding-gated judge (the fix)

Same hard rule as the other tests: any fabricated skill/employer/metric caps that variant's scores, regardless of fluency.

In [3]:
print_table(grounded, "grounded_judge_scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       1           2           2           2           1.75
Software Engineer     B       5           4           5           4           4.50
Software Engineer     C       5           3           5           3           4.00
Machine Learning      A       1           2           2           2           1.75
Machine Learning      B       1           2           2           2           1.75
Machine Learning      C       5           1           5           5           4.00
DevOps Engineer       A       1           2           2           2           1.75
DevOps Engineer       B       1           2           2           2           1.75
DevOps Engineer       C       5           4           5           5           4.75
Database Administrato A       5           4           5           4           4.50
Databa

In [4]:
# Fabrications the grounding-gated judge actually caught, per variant
for entry in grounded:
    fab = entry["grounded_judge_scores"].get("fabrications", {})
    if any(fab.get(v) for v in "ABC"):
        print(entry["job_title"], "@", entry["company_name"])
        for v in "ABC":
            items = [i for i in (fab.get(v) or []) if i]
            if items:
                print(f"  {v}: {items}")
        print()

Software Engineer @ WestGate Networks
  A: ['Java', 'Spring', 'SQL (Sybase/Oracle)', 'caching applications']

Machine Learning @ LumaCore Data
  A: ['data analysis and visualization using React and JavaScript libraries', 'data analysis and optimization techniques']
  B: ['D3.js', 'Chart.js', 'Jupyter Notebooks', 'Pandas', 'scikit-learn']

DevOps Engineer @ CloudHarbor Labs
  A: ['Python', 'C#', 'Shell/Powershell', 'Bash', 'Jenkins', 'Chef', 'PCI compliance']
  B: ['C#', 'Python', 'PowerShell', 'Shell Scripting', 'Jenkins', 'Chef', 'PCI compliance']

Database Administrator @ Solstice Digital
  B: ['Financial management: Managed budgets, optimized resource allocation, and reduced costs']

Full Stack Developer @ InnoWave Networks
  A: ['Java', 'Postgres', 'Redis', 'RabbitMQ', 'Elasticsearch']
  B: ['Java', 'Postgres', 'Redis', 'RabbitMQ', 'Elasticsearch']



## Findings

**1. A strong, real, narrow background is just as easy to over-claim from as a weak one.**

The other three resumes tested candidates with limited, absent, or mismatched experience. This one tests the opposite case: genuinely deep, real seniority — just scoped to one stack (front-end). The result is the same failure pattern: A and B don't stay within the candidate's real expertise boundary, they invent whatever the job description asks for, on 4 of 5 jobs each. Depth of real experience doesn't protect against fabrication; only an explicit anti-fabrication design does.

**2. C's honesty here has a real cost — the lowest personalization score of any resume test.**

C's grounding-gated overall (4.40) is its highest score across all four resume tests, but its personalization subscore (3.0) reflects a real trade-off: staying strictly truthful about a narrow specialty means several tailored resumes look thin against jobs outside that specialty. That's a defensible design choice (a hiring manager should see an honest fit signal, not an inflated one), but worth naming directly rather than only reporting the win.

**3. This resume produced the clearest contrast between systems of all four tests.**

The naive-judge gap between C and A/B (4.00 vs 2.90/3.20) and the grounding-gated gap (4.40 vs 2.30/2.30) are both the widest margins seen across all four resumes. A senior, specialized candidate facing a wide spread of job types is close to a worst-case stress test for fabrication-prone systems, and correspondingly the best case for demonstrating why the anti-fabrication design matters.

**4. As a Stage 7 finding (this resume, no retest needed):**

| Step | What happened |
|---|---|
| Input | Same A/B/C systems (fixed code already in place), tested on a senior candidate with deep but narrowly-scoped real experience |
| Expected | A/B fabricate to bridge domain gaps; C stays fabrication-free by design |
| Actual | Confirmed — A and B each fabricated on 4/5 jobs (their worst rate across all four resumes); C stayed at zero fabrications |
| Implication | The anti-fabrication fix generalizes across very different candidate profiles: no real experience (fresh grad), unrelated real experience (graphic design/retail), and deep-but-narrow real experience (this resume) all produce the same result — C stays clean, A/B do not |